In [22]:
!pip install transformers torch pandas openpyxl scikit-learn accelerate -q
import torch, pandas as pd, numpy as np, re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [24]:
FILE_PATH = "/content/Данные из резюме.xlsx"
df = pd.read_excel(FILE_PATH)

# Подготовка всех кандидатов
candidates = []
for idx, row in df.iterrows():
    # Создаём текстовое представление для TF-IDF
    text_parts = []
    if pd.notna(row.get('title')):
        text_parts.append(str(row['title']))
    if pd.notna(row.get('skills')):
        skills = row['skills']
        if isinstance(skills, list):
            text_parts.append(' '.join(skills))
        else:
            text_parts.append(str(skills))
    if pd.notna(row.get('work_exp_descr')):
        text_parts.append(str(row['work_exp_descr']))

    candidates.append({
        'id': row.get('id', idx),
        'title': str(row.get('title', 'N/A'))[:100],
        'work_experience': row.get('work_experience', 0),
        'skills': row.get('skills', []),
        'work_exp_descr': str(row.get('work_exp_descr', 'N/A'))[:500],
        'search_text': ' '.join(text_parts)  # Для TF-IDF
    })

In [25]:
JOB_DESCRIPTION = """
Обязанности:
-Разработка и внедрение моделей NLP для различных областей.
-Разработка и внедрение кейсов использования LLM для различных областей.
-Разработка моделей компьютерного зрения для задач создания видео-контента.
-Оптимизация моделей машинного обучения для эффективного использования (например performance или real-time).
-Совместная работа с командой для внедрения моделей в технологический стек продуктов кампании.

Требования:
-Опыт работы с NLP или LLM моделями в реальных задачах.
-Опыт работы на реальных коммерческих проектах с области CV и ML/DL или NLP.
-Понимание основ компьютерного зрения, обработки изображений и работы с текстом.
-Опыт работы с TensorFlow, PyTorch, OpenCV и др.

Будет плюсом:
-Опыт работы с opensource LLM моделями локально.
-Опыт работы с нейронными сетями для анализа и обработки видео.


"""

# TF-IDF для быстрого отсева всех резюме
corpus = [c['search_text'] for c in candidates] + [JOB_DESCRIPTION]
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), stop_words=None)
tfidf_matrix = tfidf.fit_transform(corpus)

# Косинусное сходство вакансии со всеми резюме
job_vector = tfidf_matrix[-1:]
resume_vectors = tfidf_matrix[:-1]
similarities = cosine_similarity(job_vector, resume_vectors)[0]

# топ-30 для демо LLM оценки
top_30_idx = np.argsort(similarities)[::-1][:30]
top_30_candidates = [candidates[i] for i in top_30_idx]

In [30]:
def evaluate_candidate_llm(candidate, JOB_DESCRIPTION, model, tokenizer):
    skills_str = candidate['skills'] if isinstance(candidate['skills'], str) else ', '.join(str(s) for s in candidate['skills'] if s)
    descr_str = str(candidate['work_exp_descr'])[:600]

    prompt = f"""Ты — технический рекрутер. Оцени кандидата на соответствие вакансии.

ВАКАНСИЯ:
{JOB_DESCRIPTION}

ПРОФИЛЬ КАНДИДАТА:
Должность: {candidate['title']}
Стаж: {candidate['work_experience']} лет
Навыки: {skills_str}
Опыт/Задачи: {descr_str}

ПРАВИЛА АНАЛИЗА:
1. Из текста вакансии самостоятельно выдели ключевые требования: минимальный стаж, основной стек/навыки, профиль задач и желательные квалификации.
2. Сравни требования с профилем кандидата. Учитывай смысловое соответствие и контекст выполненных задач, а не только точные совпадения слов.
3. Если в вакансии явно указан минимальный стаж, а у кандидата он меньше — ставь 1 или 2.
4. Шкала оценки (1-5):
5 — Полное совпадение: стаж соответствует, стек и задачи полностью релевантны.
4 — Сильное совпадение: основные требования выполнены, есть 1-2 некритичных различия с описанием вакансии.
3 — Частичное совпадение: Должность/стаж/навыки присутствуют, но контекст или глубина отличаются и минимальное совпадение.
2 — Слабое совпадение: смежный опыт кандидата, но ключевые требования вакансии не выполнены.
1 — Не подходит: явное несоответствие стажу или критическим навыкам.

ОТВЕТЬ СТРОГО В ФОРМАТЕ (2 строки, без пояснений, без кода и markdown):
БАЛЛ: [число от 1 до 5]
ОБОСНОВАНИЕ: 1-2 предложения с конкретными фактами, что совпало или чего не хватило."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    text = raw.replace(prompt, "").strip()

    score = 3
    is_match = False
    explanation = ""
    # explanation = "Недостаточно данных для анализа."

    for line in text.split('\n'):
        line = line.strip()
        if not line: continue
        up = line.upper()
        if "БАЛЛ:" in up:
            nums = re.findall(r'\d+', line.split("БАЛЛ:")[-1])
            if nums: score = min(5, max(1, int(nums[0])))
        elif "ПОДХОДИТ:" in up:
            is_match = "ДА" in up.split("ПОДХОДИТ:")[-1] or score >= 4
        elif "ОБОСНОВАНИЕ:" in up:
            explanation = line.split("ОБОСНОВАНИЕ:")[-1].strip()

    # Финальная очистка от артефактов генерации
    explanation = re.sub(r'[\[\]{}#`*>]', '', explanation).strip()
    if len(explanation) > 15: explanation = explanation[:117] + "..."
    if len(explanation) < 15: explanation = "Оценка на основе требований вакансии и профиля кандидата."

    return {
        'id': candidate['id'],
        'title': candidate['title'],
        'work_experience': candidate['work_experience'],
        'score': score,
        'is_match': is_match,
        'explanation': explanation
    }

# Запуск оценки
evaluated = []
for i, c in enumerate(top_30_candidates, 1):
    res = evaluate_candidate_llm(c, JOB_DESCRIPTION, model, tokenizer)
    evaluated.append(res)
    print(f"{i}. {res['title'][:45]} | {res['score']}/5")

1. Data scientist/Machine learning engineer | 3/5
2. Программист | 4/5
3. Программист 1C | 2/5
4. IOS разработчик | 4/5
5. Data Scientist Computer Vision | 3/5
6. Java backend developer | 4/5
7. Программист 1C 7.7 | 2/5
8. Аналитик данных | 3/5
9. Инженер по электрооборудованию автомобилей (A | 3/5
10. NLP-разработчик | 4/5
11. Руководитель IT-отдела | 4/5
12. Программист 1C | 3/5
13. Data Scientist | 4/5
14. Machine Learning Engineer | 4/5
15. Аналитик данных | 3/5
16. Разработчик Java/Scala | 2/5
17. Head of Data Science/ML | 3/5
18. IOS разработчик | 3/5
19. Аналитик данных | 4/5
20. Backend Разработчик | 4/5
21. Ведущий программист 1C | 3/5
22. MLOps / ML Engineer | 3/5
23. Python developer | 3/5
24. Руководитель IT проектов | 5/5
25. Программист 1C | 2/5
26. Ведущий инженер отдела технической поддержки | 4/5
27. Программист Python | 3/5
28. Data Scientist / Computer Vision Developer (R | 5/5
29. Data scientist | 4/5
30. Специалист по компьютерному зрению/Computer v | 3/5


In [31]:
# Ранжирование: сначала по баллу ИИ (убывание), при равенстве — по опыту
ranked = sorted(evaluated, key=lambda x: (x['score'], x['work_experience']), reverse=True)
top_5 = ranked[:5]

print("ТОП-5")
for i, c in enumerate(top_5, 1):
    print(f"\n{i}. ID: {c['id']}")
    print(f"   Должность: {c['title']}")
    print(f"   Опыт: {c['work_experience']} лет | Оценка ИИ: {c['score']}/5")
    print(f"   Обоснование: {c['explanation']}")

ТОП-5

1. ID: d9cc2ed500080d489b0039ed1f7939556d6a73
   Должность: Руководитель IT проектов
   Опыт: 8.67 лет | Оценка ИИ: 5/5
   Обоснование: Кандидат работает уже больше 8 лет как руководитель IT проектов, который также владеет необходимыми н...

2. ID: 95104816000833112e0039ed1f6e496c507577
   Должность: Data Scientist / Computer Vision Developer (RU)
   Опыт: 2.17 лет | Оценка ИИ: 5/5
   Обоснование: Кандидат работает уже два года как Junior Data Scientist, где...

3. ID: fc18669100001d421a0039ed1f736563726574
   Должность: Руководитель IT-отдела
   Опыт: 19.05 лет | Оценка ИИ: 4/5
   Обоснование: Оценка на основе требований вакансии и профиля кандидата.

4. ID: dcff6a170007f7732e0039ed1f79716b53545a
   Должность: Data scientist
   Опыт: 11.92 лет | Оценка ИИ: 4/5
   Обоснование: Оценка на основе требований вакансии и профиля кандидата.

5. ID: d35b54040008dbc2870039ed1f7a69434c3474
   Должность: Machine Learning Engineer
   Опыт: 6.05 лет | Оценка ИИ: 4/5
   Обоснование: Кандидат 